# Observatorio de Obras Públicas · 03 Dashboard

Toma `datos/obras_limpias.csv` (paso 02) y genera un dashboard web con:

- Inversión total del Pacífico caucano, por municipio y **por habitante** (con la población oficial del DANE).
- Línea de tiempo de la inversión por municipio.
- Mapa satelital de los tres municipios.
- Filtros por municipio, contratista, tipo de obra, estado (terminada, en ejecución...), año y alertas.
- Tabla de contratos con enlace a SECOP.

El resultado es **un solo archivo**: `docs/index.html` (la carpeta `docs/` es la que publica GitHub Pages). Ábrelo con doble clic en el navegador; funciona sin internet, salvo las fuentes tipográficas.

**Cómo ejecutarlo:** pon el cursor en una celda y pulsa **Shift + Enter**. La primera vez descarga la población y las imágenes satelitales (1–2 minutos); después usa la copia guardada en `datos/`.

## 1 · Configuración

Los tres municipios del Pacífico caucano, con su código DANE, la ubicación de la cabecera y el color que tendrán en todo el dashboard.

In [1]:
import os, io, json, time, base64, math, datetime
import requests
import pandas as pd
import duckdb

pd.options.display.float_format = lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:,.1f}"

MUNICIPIOS = [
    # nombre en los datos, nombre para mostrar, código DANE, cabecera (lat, lon), color
    {"nombre": "GUAPI",          "titulo": "Guapi",          "codigo": "19318", "lat": 2.5708, "lon": -77.8826, "color": "--m1"},
    {"nombre": "TIMBIQUI",       "titulo": "Timbiquí",       "codigo": "19809", "lat": 2.7705, "lon": -77.6643, "color": "--m2"},
    {"nombre": "LOPEZ DE MICAY", "titulo": "López de Micay", "codigo": "19418", "lat": 2.8476, "lon": -77.2478, "color": "--m3"},
]
HOY = pd.Timestamp(datetime.date.today())
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
CORTE_TEXTO = f"{HOY.day} de {MESES[HOY.month - 1]} de {HOY.year}"

os.makedirs("datos/cache", exist_ok=True)
os.makedirs("dashboard", exist_ok=True)
os.makedirs("docs", exist_ok=True)
print("Fecha de corte:", CORTE_TEXTO)

Fecha de corte: 30 de octubre de 2026


## 2 · Estado de cada obra

SECOP no dice directamente si una obra está terminada. Se deduce así:

| Estado | Regla |
|---|---|
| **Terminada** | El contrato figura como *Liquidado*, *Terminado* o *Cerrado*. |
| **En contratación** | Figura como *Convocado*: hay proceso pero aún no hay contrato. |
| **Plazo vencido** | La fecha de fin ya pasó y el contrato **no** figura liquidado. |
| **En ejecución** | Todos los demás: firmados y con la fecha de fin por delante. |

In [2]:
obras = pd.read_csv("datos/obras_limpias.csv", parse_dates=["fecha_firma", "fecha_inicio", "fecha_fin"])

def estado_obra(fila):
    if fila["estado"] in ("Liquidado", "Terminado", "Cerrado"):
        return "Terminada"
    if fila["estado"] == "Convocado":
        return "En contratación"
    if pd.notna(fila["fecha_fin"]) and fila["fecha_fin"] < HOY:
        return "Plazo vencido"
    return "En ejecución"

obras["estado_obra"] = obras.apply(estado_obra, axis=1)
pd.crosstab(obras["estado"], obras["estado_obra"], margins=True, margins_name="Total")

estado_obra,En contratación,En ejecución,Plazo vencido,Terminada,Total
estado,,,,,
Adjudicado,0,3,52,0,55
Celebrado,0,4,164,0,168
Convocado,50,0,0,0,50
En ejecución,0,1,3,0,4
Liquidado,0,0,0,84,84
Modificado,0,2,0,0,2
Total,50,10,219,84,363


## 3 · Población oficial (DANE)

Para calcular la inversión **por habitante** se usan las *Proyecciones de población municipal 2018–2042* del DANE (basadas en el Censo 2018). El archivo pesa unos 4 MB y se guarda en `datos/cache/` para no descargarlo cada vez.

In [3]:
URL_DANE = "https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion/Municipal/PPED-AreaMun-2018-2042_VP.xlsx"
archivo_dane = "datos/cache/dane_poblacion_municipal.xlsx"
if not os.path.exists(archivo_dane):
    r = requests.get(URL_DANE, timeout=300, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    open(archivo_dane, "wb").write(r.content)

dane = pd.read_excel(archivo_dane, sheet_name="PobMunicipalxÁrea", header=7, dtype={"MPIO": str})
codigos = {m["codigo"]: m["nombre"] for m in MUNICIPIOS}
anios = sorted(obras["anio"].unique())

poblacion = (dane[(dane["ÁREA GEOGRÁFICA"] == "Total") & dane["MPIO"].isin(codigos) & dane["AÑO"].isin(anios)]
             .assign(municipio=lambda d: d["MPIO"].map(codigos), anio=lambda d: d["AÑO"].astype(int))
             [["municipio", "anio", "TOTAL"]].rename(columns={"TOTAL": "habitantes"}))
poblacion.pivot(index="municipio", columns="anio", values="habitantes")

anio,2022,2023,2024,2025,2026
municipio,,,,,
GUAPI,"28,736","28,979","29,275","29,530","29,766"
LOPEZ DE MICAY,"19,198","19,325","19,491","19,624","19,757"
TIMBIQUI,"28,045","28,538","29,010","29,477","29,935"


## 4 · Inversión total, por municipio y por habitante

**Inversión** = valor firmado de los contratos de obra (no lo pagado).
**Por habitante** = valor de cada año ÷ habitantes de ese año, sumado en todo el periodo.

In [4]:
por_anio = duckdb.sql("""
    SELECT o.municipio, o.anio,
           COUNT(*)                     AS contratos,
           SUM(o.valor)                 AS inversion,
           ANY_VALUE(p.habitantes)      AS habitantes,
           SUM(o.valor) / ANY_VALUE(p.habitantes) AS por_habitante
    FROM obras o JOIN poblacion p USING (municipio, anio)
    GROUP BY ALL
""").df()

resumen = duckdb.sql("""
    SELECT municipio,
           CAST(SUM(contratos) AS INT) AS contratos,
           SUM(inversion)      AS inversion,
           SUM(por_habitante)  AS por_habitante,
           ROUND(100 * SUM(inversion) / SUM(SUM(inversion)) OVER (), 1) AS porcentaje
    FROM por_anio GROUP BY municipio ORDER BY inversion DESC
""").df()

pob_region = poblacion.groupby("anio")["habitantes"].sum()
region_pc = (obras.groupby("anio")["valor"].sum() / pob_region).sum()
print(f"Inversión total en el Pacífico caucano: ${obras['valor'].sum() / 1e9:,.1f} mil millones")
print(f"Por habitante (todo el periodo):        ${region_pc:,.0f}")
resumen

Inversión total en el Pacífico caucano: $118.0 mil millones
Por habitante (todo el periodo):        $1,504,554


,municipio,contratos,inversion,por_habitante,porcentaje
0,LOPEZ DE MICAY,277,"73,717,604,553","3,759,213",62.5
1,GUAPI,73,"28,111,848,894","956,409",23.8
2,TIMBIQUI,13,"16,124,103,770","551,883",13.7


### 4.1 · Línea de tiempo por municipio (mil millones de pesos)

In [5]:
(por_anio.pivot(index="anio", columns="municipio", values="inversion") / 1e9).round(1)

municipio,GUAPI,LOPEZ DE MICAY,TIMBIQUI
anio,,,
2022,7.6,14.2,3.8
2023,0.5,2.0,1.5
2024,0.8,3.2,1.6
2025,7.1,7.5,0.9
2026,12.0,46.8,8.3


## 5 · Límites de los municipios

Se descargan de OpenStreetMap (servicio Nominatim) y se guardan en `datos/cache/`.

In [6]:
def limite(m):
    archivo = f"datos/cache/limite_{m['codigo']}.json"
    if not os.path.exists(archivo):
        r = requests.get("https://nominatim.openstreetmap.org/search", timeout=60,
                         headers={"User-Agent": "observatorio-obras-portfolio/1.0"},
                         params={"q": f"{m['titulo']}, Cauca, Colombia", "format": "geojson",
                                 "polygon_geojson": 1, "polygon_threshold": 0.003, "limit": 5})
        r.raise_for_status()
        open(archivo, "w").write(r.text)
        time.sleep(1)  # Nominatim pide máximo una consulta por segundo
    features = json.load(open(archivo))["features"]
    return next(f for f in features if f["properties"].get("category") == "boundary")

for m in MUNICIPIOS:
    f = limite(m)
    m["geometria"] = f["geometry"]
    m["bbox"] = f["bbox"]
    print(f"{m['titulo']:15} {f['geometry']['type']:12} bbox {[round(v, 2) for v in f['bbox']]}")

BOUNDS = [min(m["bbox"][0] for m in MUNICIPIOS), min(m["bbox"][1] for m in MUNICIPIOS),
          max(m["bbox"][2] for m in MUNICIPIOS), max(m["bbox"][3] for m in MUNICIPIOS)]

Guapi           Polygon      bbox [-77.93, 2.13, -77.38, 2.71]
Timbiquí        Polygon      bbox [-77.8, 2.38, -77.2, 2.94]
López de Micay  Polygon      bbox [-77.7, 2.67, -76.78, 3.25]


## 6 · Imágenes satelitales

El mapa usa las imágenes satelitales de Esri (World Imagery). Se descargan solo los cuadros (*tiles*) que cubren la zona, en los niveles de zoom 9 a 11, y se **guardan dentro del dashboard**. Así el mapa funciona aunque el sitio donde publiques no permita cargar imágenes de otros servidores.

In [7]:
ZOOM_MIN, ZOOM_MAX = 9, 11
URL_TILE = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"

def tile_xy(lon, lat, z):
    n = 2 ** z
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n)
    return x, y

tiles, margen = {}, 0.15
for z in range(ZOOM_MIN, ZOOM_MAX + 1):
    x0, y0 = tile_xy(BOUNDS[0] - margen, BOUNDS[3] + margen, z)
    x1, y1 = tile_xy(BOUNDS[2] + margen, BOUNDS[1] - margen, z)
    for x in range(x0, x1 + 1):
        for y in range(y0, y1 + 1):
            archivo = f"datos/cache/tiles/{z}/{x}/{y}.jpg"
            if not os.path.exists(archivo):
                os.makedirs(os.path.dirname(archivo), exist_ok=True)
                r = requests.get(URL_TILE.format(z=z, x=x, y=y), timeout=60)
                r.raise_for_status()
                open(archivo, "wb").write(r.content)
            tiles[f"{z}/{x}/{y}"] = "data:image/jpeg;base64," + base64.b64encode(open(archivo, "rb").read()).decode()

peso = sum(len(v) for v in tiles.values()) / 1e6
print(f"{len(tiles)} imágenes satelitales · {peso:.1f} MB dentro del dashboard")

132 imágenes satelitales · 2.3 MB dentro del dashboard


## 7 · Datos para el dashboard

Se arma un solo paquete JSON con los contratos, los municipios (con límites y color) y la población por año.

In [8]:
titulos = {m["nombre"]: m["titulo"] for m in MUNICIPIOS}
ALERTAS = ["valor_atipico", "objeto_repetido", "sin_firmar", "adicion_alta"]

contratos = [{
    "id": f.id,
    "fuente": f.fuente,
    "municipio": f.municipio,
    "municipioTitulo": titulos[f.municipio],
    "anio": int(f.anio),
    "fecha": f.fecha_firma.strftime("%Y-%m-%d"),
    "sector": f.sector,
    "objeto": f.objeto,
    "contratista": f.contratista_unico,
    "estado": f.estado_obra,
    "valor": float(f.valor),
    "alertas": [a for a in ALERTAS if getattr(f, a)],
    "url": f.url if isinstance(f.url, str) and f.url.startswith("http") else "",
} for f in obras.itertuples()]

datos = {
    "contratos": contratos,
    "municipios": [{k: m[k] for k in ("nombre", "titulo", "lat", "lon", "color", "geometria")} for m in MUNICIPIOS],
    "poblacion": {mun: {int(a): int(h) for a, h in zip(g["anio"], g["habitantes"])} for mun, g in poblacion.groupby("municipio")},
    "bounds": BOUNDS,
    "maxTileZoom": ZOOM_MAX,
    "anioCorte": int(HOY.year),
    "corteTexto": CORTE_TEXTO,
}
print(f"{len(contratos)} contratos listos para el dashboard")

363 contratos listos para el dashboard


## 8 · Generar el dashboard

Se llena la plantilla `dashboard/plantilla.html` con los datos y se guardan dos archivos:

- `docs/index.html`: página completa. Ábrela con doble clic; es la que publica GitHub Pages.
- `dashboard/dashboard.html`: la misma página sin la cabecera HTML, para publicarla como Artifact en claude.ai.

In [9]:
URL_LEAFLET_CSS = "https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"
archivo_css = "datos/cache/leaflet.min.css"
if not os.path.exists(archivo_css):
    open(archivo_css, "w").write(requests.get(URL_LEAFLET_CSS, timeout=60).text)

def a_js(objeto):
    # "</" se escapa para que ningún texto pueda cerrar la etiqueta <script>
    return json.dumps(objeto, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")

pagina = (open("dashboard/plantilla.html", encoding="utf-8").read()
          .replace("/*__LEAFLET_CSS__*/", open(archivo_css).read())
          .replace("/*__DATOS__*/", a_js(datos))
          .replace("/*__TILES__*/", a_js(tiles)))

open("dashboard/dashboard.html", "w", encoding="utf-8").write(pagina)
open("docs/index.html", "w", encoding="utf-8").write(
    '<!doctype html>\n<html lang="es">\n<head>\n<meta charset="utf-8">\n'
    '<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n'
    '<style>[hidden]{display:none!important} body{margin:0}</style>\n</head>\n<body>\n'
    + pagina + "\n</body>\n</html>\n")

print(f"Guardado: docs/index.html ({os.path.getsize('docs/index.html') / 1e6:.1f} MB)")

Guardado: docs/index.html (2.6 MB)
